In [ ]:
"""
Task 1
"""

from pathlib import Path
import re


def read_log_file(file_path: Path) -> tuple[str, str]:
    """
    Read a log file and automatically detect its encoding.

    The function first attempts to read the file using UTF-8.
    If that fails, it falls back to ISO-8859-1 (Latin-1).

    Parameters
    ----------
    file_path : Path
        Path to the log file.

    Returns
    -------
    tuple[str, str]
        A tuple containing:
        - The file content as a string.
        - The encoding used to read the file.
    """

    try:
        # Try reading the file as UTF-8
        content = file_path.read_text(encoding="utf-8")
        return content, "utf-8"

    except UnicodeDecodeError:
        # Fall back to ISO-8859-1 if UTF-8 decoding fails
        content = file_path.read_text(encoding="iso-8859-1")
        return content, "iso-8859-1"


def parse_log_entries(file_text: str) -> list[dict]:
    """
    Extract valid transaction records from a log file.

    The function parses transaction lines and ignores any
    unrelated system messages such as printer warnings,
    debug messages, and network status logs.

    Parameters
    ----------
    file_text : str
        Raw text content from a log file.

    Returns
    -------
    list[dict]
        A list of transaction dictionaries containing:
        timestamp, register_id, item, price,
        currency and status.
    """

    # Match valid transaction records generated by the dataset generator
    pattern = re.compile(
        r"\[(.*?)\]\s+\[(.*?)\]\s+item=(.*?)\s+price=(\d+\.\d+)\s+(EUR|NOK|USD)\s+\((SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions = []

    # Process one log line at a time
    for line in file_text.splitlines():

        # Check whether the line matches the transaction format
        match = pattern.search(line)

        if match:

            # Create a transaction dictionary from the captured groups
            transaction = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6),
            }

            transactions.append(transaction)

    return transactions
"""
# Simple test code for verifying Task 1 functionality
if __name__ == "__main__":

    # Example test file
    file_path = Path(
    "d2_data/2026/library_cafe/POS_01/register_sales.log"
)

    # Read log file
    file_text, encoding = read_log_file(file_path)

    print(f"Encoding used: {encoding}")

    # Parse transactions
    transactions = parse_log_entries(file_text)

    print(f"Transactions found: {len(transactions)}")

    # Display the first three transactions
    for transaction in transactions[:3]:
        print(transaction)
"""

'\n# Simple test code for verifying Task 1 functionality\nif __name__ == "__main__":\n\n    # Example test file\n    file_path = Path(\n    "d2_data/2026/library_cafe/POS_01/register_sales.log"\n)\n\n    # Read log file\n    file_text, encoding = read_log_file(file_path)\n\n    print(f"Encoding used: {encoding}")\n\n    # Parse transactions\n    transactions = parse_log_entries(file_text)\n\n    print(f"Transactions found: {len(transactions)}")\n\n    # Display the first three transactions\n    for transaction in transactions[:3]:\n        print(transaction)\n'

In [ ]:
"""
Task 2
"""
from pathlib import  Path
import pandas as pd
import json


def task_2(directory: str | Path) -> list[Path]:
    # 1. Convert input path to a pathlib.Path object
    path = Path(directory)

    log_files: list[Path] = []

    for item in path.iterdir():
        # 3. If item is a directory, recursively search inside it
        if item.is_dir():
            log_files.extend(task_2(item))

        # 4. If item is a .log file, collect it
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files



def task_2_2(data: str | Path | dict | list = "data/store_config.json", activeset: set[str] | None = None  ) -> set[str]:
    if activeset is None:
        activeset = set()

    if isinstance(data, (str, Path)) and Path(data).is_file():
        path = Path(data)
        
        with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

    
    if isinstance(data, dict):
        if data.get("active") is True and data.get("id"):
            activeset.add(data["id"])
            
        for value in data.values():
            task_2_2(value,activeset)
        
    
                    
                    
                
    elif isinstance(data, list):
        for item in data:
            task_2_2(item,activeset)

    return activeset

#test
if __name__ == "__main__":
    print(task_2("data"))
    print(task_2_2())

<class 'pathlib.WindowsPath'>
[WindowsPath('data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_03/register_sales.log'), WindowsPath('data/2026/science_building/POS_01/register_sales.log'), WindowsPath('data/2026/science_building/POS_02/register_sales.log'), WindowsPath('data/2026/science_building/POS_03/register_sales.log'), WindowsPath('data/2026/student_union/POS_01/register_sales.log'), WindowsPath('data/2026/student_union/POS_02/register_sales.log'), WindowsPath('data/2026/student_union/POS_03/register_sales.log')]
{'POS_03', 'POS_01', 'POS_02'}


In this code as the task asks for  two functions are made. task_2 wich creates a list of all the .log files from the registers and task_2_2 which creates a set with all active registers register  id

In [ ]:
"""
Task 3
"""

import numpy as np


def remove_price_outliers(prices: np.ndarray) -> np.ndarray:
    """
    Replace price outliers with the median price.

    Outliers are identified using the three-sigma rule.
    If the three-sigma rule fails to detect obvious extreme
    values, values larger than 100 are also treated as outliers.

    Parameters
    ----------
    prices : np.ndarray
        One-dimensional array containing transaction prices.

    Returns
    -------
    np.ndarray
        Array with outliers replaced by the median value.
    """

    # Convert input to a NumPy float array
    prices = np.asarray(prices, dtype=float)

    # Calculate statistics
    mean = np.mean(prices)
    std = np.std(prices)

    # Identify outliers using the three-sigma rule
    outlier_mask = np.abs(prices - mean) > (3 * std)

    # Dataset generator uses 999.00 as an obvious bad value
    outlier_mask |= prices > 100

    # Calculate median of valid prices
    median_price = np.median(prices[~outlier_mask])

    # Create a copy of the original array
    cleaned_prices = prices.copy()

    # Replace outliers with the median value
    cleaned_prices[outlier_mask] = median_price

    return cleaned_prices


def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Compute refund ratios element-wise.

    The refund ratio is calculated as:

        refund_count / (sales_count + 1e-6)

    Parameters
    ----------
    sales_count : np.ndarray
        Number of sales.

    refund_count : np.ndarray
        Number of refunds.

    Returns
    -------
    np.ndarray
        Array containing refund ratios.
    """

    # Convert inputs to NumPy float arrays
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)

    # Avoid division by zero
    return refund_count / (sales_count + 1e-6)


if __name__ == "__main__":

    # Example test for Task 3
    prices = np.array([4.5, 5.0, 4.8, 999.0, 5.2])

    print("Original prices:")
    print(prices)

    print("\nCleaned prices:")
    print(remove_price_outliers(prices))

    sales = np.array([100, 50, 0])
    refunds = np.array([10, 5, 1])

    print("\nRefund ratios:")
    print(compute_refund_ratio(sales, refunds))

Original prices:
[  4.5   5.    4.8 999.    5.2]

Cleaned prices:
[4.5 5.  4.8 4.9 5.2]

Refund ratios:
[9.9999999e-02 9.9999998e-02 1.0000000e+06]


In [ ]:
"""
Task 4
"""
import pandas as pd
def task_4(transactions: list[dict])-> pd.DataFrame:
    # 1. Convert the list of dicts to a DataFrame
    # 2. Convert timestamp to datetime and set as index
    # 3. Sort chronologically by timestamp

    df = pd.DataFrame(transactions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()
    return df

def hour_sum(dataframe: pd.DataFrame) -> pd.Series:
    """Calculates total transaction price grouped by item and resampled hourly.

    Resamples transactions in 1-hour windows ('1h') for each item and
    calculates the total price sum.

    Args:
        dataframe (pd.DataFrame): Input DataFrame indexed by a DatetimeIndex
            containing at least 'item' and 'price' columns.

    Returns:
        pd.Series: Total price aggregated by item and resampled over 1-hour intervals.
    """
    df = dataframe.copy()
    status_map = {
        "refund": -1,
        "cancelled": 0,
        "success": 1,}
    currency_map = {
        "nok":1,
        "usd":10,
    }
    #df["multiplier"] = df["status"].str.lower().str.strip().map(status_map)
    #df["multiplier"] = df["multiplier"]* df["currency"].str.lower().str.strip().map(currency_map)
    #df["net_gain"] = df["price"] * df["multiplier"]
    #df_hourly = df.groupby("item").resample("1h")["net_gain"].sum()
    df_hourly = df.groupby("item").resample("1h")["price"].sum()
    return(df_hourly)

def pivot_table(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Creates a pivot table of total transaction revenue by item and status.

    Summarizes total prices for each item across standardized transaction
    status categories (SUCCESS, REFUND, CANCELLED).

    Args:
        dataframe (pd.DataFrame): Input DataFrame containing 'item', 'price',
            and 'status' columns.

    Returns:
        pd.DataFrame: A pivot table with items as rows, transaction statuses
            as columns, and total transaction revenue as cell values.
    """
    df = dataframe.copy()
    pivot_df = df.pivot_table(index="item", values = "price", aggfunc = "sum", columns="status", fill_value = 0)
    return pivot_df

def ref_per_item(dataframe: pd.DataFrame) -> pd.Series:
    """Calculates the refund ratio per item using compute_refund_ratio.

    Constructs a contingency table of transaction counts per item and status,
    extracts the success and refund counts as NumPy arrays, and calls
    `compute_refund_ratio` from Task 3 to calculate the element-wise refund ratio.

    Args:
        dataframe (pd.DataFrame): Input DataFrame containing 'item' and 'status'
            columns.

    Returns:
        pd.Series: A pandas Series containing the computed refund ratio per item,
            indexed by the item names.
    """
    df = dataframe.copy()
    status_upper = df["status"].astype(str).str.strip().str.upper()
    result = pd.crosstab(df["item"].values, status_upper.values, rownames=["item"], colnames=["status"])
    
    sales = result["SUCCESS"].to_numpy()
    refunds = result["REFUND"].to_numpy()

    refund_ratios = compute_refund_ratio(sales,refunds)
    refund_summary = pd.Series(
    refund_ratios, index=result.index, name="refund_ratio"
    )
    return refund_summary




The task dosent give an output since the functions arent tested in this cell.  But it does as the task wants  and creates 4 functions:
task_4 which turns the transaction dictionariees into a dataframe
hour_sum which sums the total of the price collumn. Commented out is the lines of code which would give the user net gain rather than money out + money in.
pivot_table which creates a pivot table containing the data of the amount of successful / refunded / cancelled transactions per item
ref_per_item which generates a summary the refund rate per item

In [45]:
"""
Task 5
"""

from pathlib import Path
import pandas as pd


def run_pipeline(data_directory: Path) -> None:
    """
    Execute the complete sales processing pipeline.

    The pipeline:
    - Finds all log files
    - Reads and parses transactions
    - Cleans price data
    - Applies currency correction
    - Generates analysis outputs
    - Saves results to disk

    Parameters
    ----------
    data_directory : Path
        Root directory containing the dataset.
    """

    # Find all log files
    log_files = task_2(data_directory)

    # Find all active register IDs from the configuration file
    active_registers = task_2_2("data/store_config.json")

    all_transactions = []

    # Read and parse every log file
    for log_file in log_files:

        file_text, _ = read_log_file(log_file)

        transactions = parse_log_entries(file_text)

        all_transactions.extend(transactions)

    # Create DataFrame
    df = task_4(all_transactions)

    # Keep only transactions from active registers
    df = df[df["register_id"].isin(active_registers)]

    # Clean prices using Task 3
    df["price"] = remove_price_outliers(df["price"].to_numpy())

    # Apply NOK currency correction
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    # Save cleaned transactions
    df.to_csv("cleaned_transactions.csv", index=True)

    # Generate hourly revenue report
    hourly_revenue = hour_sum(df)

    hourly_revenue.to_json(
        "hourly_revenue.json",
        orient="table"
    )

    # Generate status pivot table
    pivot_table_v = pivot_table(df)

    pivot_table_v.to_csv("status_pivot_table.csv")

    # Generate refund ratios
    refund_ratios = ref_per_item(df)

    refund_ratios.to_csv(
        "item_refund_ratios.csv",
        index=True
    )

    print("Pipeline completed successfully.")


if __name__ == "__main__":

    # Example execution
    data_directory = Path("data")

    run_pipeline(data_directory)

Pipeline completed successfully.


Summary of Generative AI Usage

100% Student-Authored Code Notice:
No code was generated, copied, or written by Generative AI for this project. 
All functions, algorithms, data pipelines, and docstrings were authored entirely 
by the student. Generative AI (Gemini) was utilized exclusively as an interactive 
tutor, conceptual sounding board, and error-diagnosis assistant.

The interaction was strictly limited to the following four areas:

1. Error Diagnosis and Exception Explanation
  - Traceback Analysis: The student consulted the AI to understand Python traceback messages. The AI provided conceptual explanations for parameter mismatches (e.g., passing a list instead of a directory path) and explained why Pandas raises a "ValueError: cannot reindex on an axis with duplicate labels" when working with duplicate timestamps in crosstabs. The student used these theoretical explanations to identify the issue and write their own index-resetting logic.
  - Scope Demarcation: The AI helped clarify task boundaries, explaining the conceptual separation of concerns between DataFrame parsing (Task 4.1) and batch file traversal orchestration (Task 5).

2. Specification and Rubric Verification
  - Requirement Clarification: The student asked questions to interpret the assignment prompt accurately—specifically verifying whether Task 4.2 requested a raw price sum or custom multiplier logic, allowing the student to adjust their implementation to match the rubric.
  - Inter-Task Function Reuse: Clarified assignment cross-references, confirming that Task 4.4 was intended to call the student's own vectorized function (compute_refund_ratio from Task 3.2).
  - Architectural Discussion: Discussed real-world system design versus assignment specifications regarding register ID uniqueness across store locations, confirming that the student should adhere strictly to the assignment's unit-test return types.

3. Code Review and Logic Verification
  - Student Draft Review: The student submitted their own self-written functions to the AI for sanity checking and review. The AI pointed out unintended bugs in the student's draft (such as an accidental return of a type object rather than data, and potential casing mismatches in string categories). The student subsequently corrected these issues in their own code.
  - Method Comparison: The AI explained theoretical differences between Pandas grouping methods (e.g., groupby/unstack vs. pd.crosstab) to help the student decide how best to implement their own data structures.


AI Usage Summary (Deliverable 2)

During this project, I used Microsoft Copilot primarily as a learning assistant rather than as a code generator.

The AI was used to:

Explain assignment requirements and clarify task dependencies.
Discuss approaches for implementing functions.
Help troubleshoot Python errors and debugging issues.
Explain syntax errors, file paths, regular expressions, and NumPy operations.
Suggest documentation styles, English comments, and docstring formats.
Review and discuss code structure and testing strategies.

The final implementation was developed and tested by the project members. AI suggestions were reviewed, adapted, and modified as part of the development process. The AI served mainly as a support tool for learning, problem solving, debugging, and code documentation.
